In [1]:
import pandas as pd

df = pd.read_csv("/Users/miriambenali/Desktop/Project-Simplon/HR-pulse-ai-platform/data/processed_data/processed_data.csv")
df.head()


,Job Title,Job Description,Company Name,Size,Type of ownership,Industry,Sector,Revenue,mean_salary
0,senior data scientist,descriptionthe senior data scientist is respon...,healthfirst3.1,1001 to 5000 employees,nonprofit organization,insurance carriers,insurance,unknown nonapplicable,11.944708
1,data scientist,"secure our nation, ignite your futurejoin the ...",mantech4.2,5001 to 10000 employees,company public,research development,business services,1 to 2 billion usd,11.944708
2,data scientist,overviewanalysis group is one of the largest i...,analysis group3.8,1001 to 5000 employees,private practice firm,consulting,business services,100 to 500 million usd,11.944708
3,data scientist,job descriptiondo you have a passion for data ...,inficon3.5,501 to 1000 employees,company public,electrical electronic manufacturing,manufacturing,100 to 500 million usd,11.944708
4,data scientist,data scientistaffinity solutions marketing cl...,affinity solutions2.9,51 to 200 employees,company private,advertising marketing,business services,unknown nonapplicable,11.944708


In [2]:
import numpy as np

In [3]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')


all_embeddings = []

/Users/miriambenali/Desktop/Project-Simplon/HR-pulse-ai-platform/myvenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/miriambenali/Desktop/Project-Simplon/HR-pulse-ai-platform/myvenv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
import numpy as np

emb = model.encode(df['Job Title'].astype(str).tolist(), show_progress_bar=True)
emb_df = pd.DataFrame(emb, columns=[f"jobtitle_emb_{i}" for i in range(emb.shape[1])])
all_embeddings.append(emb_df)
print("✅ Job Title done:", emb_df.shape)

Batches: 100%|██████████| 21/21 [00:05<00:00,  4.17it/s]

✅ Job Title done: (672, 384)


In [5]:
emb = model.encode(df['Job Description'].astype(str).tolist(), 
                   show_progress_bar=True, 
                   batch_size=16)  # batch plus petit pour éviter crash mémoire
emb_df = pd.DataFrame(emb, columns=[f"jobdesc_emb_{i}" for i in range(emb.shape[1])])
all_embeddings.append(emb_df)
print("✅ Job Description done:", emb_df.shape)

Batches: 100%|██████████| 42/42 [00:41<00:00,  1.00it/s]

✅ Job Description done: (672, 384)


In [6]:
for col in ['Company Name', 'Size', 'Type of ownership', 'Industry', 'Sector', 'Revenue']:
    emb = model.encode(df[col].astype(str).tolist(), show_progress_bar=True)
    emb_df = pd.DataFrame(emb, columns=[f"{col.replace(' ','_')}_emb_{i}" for i in range(emb.shape[1])])
    all_embeddings.append(emb_df)
    print(f"✅ {col} done")

Batches: 100%|██████████| 21/21 [00:05<00:00,  4.10it/s]


✅ Company Name done


Batches: 100%|██████████| 21/21 [00:02<00:00, 10.26it/s]


✅ Size done


Batches: 100%|██████████| 21/21 [00:01<00:00, 12.16it/s]


...

✅ Industry done


Batches: 100%|██████████| 21/21 [00:01<00:00, 12.64it/s]


✅ Sector done


Batches: 100%|██████████| 21/21 [00:02<00:00,  9.60it/s]

✅ Revenue done


In [8]:
#assembler 
df_embeddings = pd.concat(all_embeddings, axis=1)
df_embeddings['mean_salary'] = df['mean_salary'].values

print(df_embeddings.shape)


(672, 3073)


In [11]:
# Sauvegarder immédiatement !
path = "/Users/miriambenali/Desktop/Project-Simplon/HR-pulse-ai-platform/data/processed_data/embeddings_dataset.csv"

df_embeddings.to_csv(path, index=False)